# Papermill driver for preprocessing (new CAP paths)

Runs `001_Preprocessing.ipynb` once for each included sample in the frozen cohort release.
Outputs are stored under the configured `CAP_DATA_ROOT/runs/<CAP_RUN_ID>/01_preprocessing`
stage. This driver does not create a new run ID and does not skip samples just because an
executed notebook already exists.

- Set `RUN_DEMO = True` in the settings cell to rerun only the two demo samples.
- A full run resets the `01_Preprocessing` stage before starting. A demo run only replaces
  the selected samples' outputs and writes a separate demo QC summary.
- Failed samples retain their partial notebook and log; the combined QC summary is written
  only when every selected sample succeeds.

**Destructive behavior:** a full run deletes the existing `01_Preprocessing` stage for the
current `CAP_RUN_ID` before rerunning it. Other stages and the `00_cohort` stage are left alone.


## Run settings

In [1]:
RUN_DEMO = False
DEMO_IDS = [
    "GSM4909292_HER2-MH0069",
    "GSM4909272_N-MH0021-Total",
]
COHORT_RELEASE_NAME = "cohort_release_2026-11-13.csv"

# None uses the kernelspec recorded in 001_Preprocessing.ipynb.
# Set this to the registered kernelspec name only if it differs.
KERNEL_NAME = None


## Resolve paths and validate inputs

In [2]:
from pathlib import Path
import shutil

import pandas as pd
import papermill as pm

from cap.paths import get_paths
from cap.provenance import validate_frozen_inputs, write_run_provenance

paths = get_paths()

raw_dir = paths.raw_counts_dir
features_file = paths.features_file
policy_file = paths.frozen_dir / "gene_policy.json"
cohort_file = paths.root / "config" / COHORT_RELEASE_NAME
preprocessing_notebook = paths.root / "notebooks" / "001_Preprocessing.ipynb"

paths.require_dir(raw_dir, "GSE161529 filtered count directory")
paths.require_file(features_file, "shared features.tsv")
paths.require_file(policy_file, "frozen gene policy")
paths.require_file(cohort_file, "frozen cohort release")
paths.require_file(preprocessing_notebook, "001 preprocessing notebook")
validate_frozen_inputs(paths)

def display_path(path):
    """Return a project-relative display path without local absolute paths."""
    resolved = Path(path).resolve()
    for root, label in (
        (paths.data_root, "<CAP_DATA_ROOT>"),
        (paths.root, "<REPOSITORY>"),
    ):
        try:
            relative = resolved.relative_to(Path(root).resolve())
            return f"{label}/{relative.as_posix()}"
        except ValueError:
            continue
    return f"<external>/{resolved.name}"


## Select included samples

In [3]:
cohort = pd.read_csv(cohort_file)

if "sample_id" not in cohort.columns:
    raise KeyError(
        f"Frozen cohort must contain a 'sample_id' column; found: {list(cohort.columns)}"
    )

cohort["sample_id"] = cohort["sample_id"].astype("string").str.strip()
if cohort["sample_id"].isna().any() or cohort["sample_id"].eq("").any():
    raise ValueError("Frozen cohort contains a missing or empty sample_id.")
if cohort["sample_id"].duplicated().any():
    duplicates = cohort.loc[cohort["sample_id"].duplicated(), "sample_id"].tolist()
    raise ValueError(f"Duplicate sample_id values in frozen cohort: {duplicates}")

if "include" in cohort.columns:
    include_map = {
        "true": True,
        "false": False,
        "1": True,
        "0": False,
        "yes": True,
        "no": False,
    }
    include = (
        cohort["include"].astype("string").str.strip().str.lower().map(include_map)
    )
    if include.isna().any():
        bad_values = cohort.loc[include.isna(), "include"].drop_duplicates().tolist()
        raise ValueError(f"Unrecognised values in cohort 'include' column: {bad_values}")
    eligible_ids = cohort.loc[include, "sample_id"].astype(str).tolist()
else:
    # A release containing only accepted samples does not need an include column.
    eligible_ids = cohort["sample_id"].astype(str).tolist()

if RUN_DEMO:
    missing_demo_ids = sorted(set(DEMO_IDS).difference(eligible_ids))
    if missing_demo_ids:
        raise ValueError(
            f"Demo sample(s) are not included in the frozen cohort: {missing_demo_ids}"
        )
    sample_ids = list(DEMO_IDS)
else:
    sample_ids = eligible_ids

if not sample_ids:
    raise ValueError("No samples were selected for preprocessing.")

print(f"Frozen cohort: {display_path(cohort_file)}")
print(f"Selected samples: {len(sample_ids)}" + (" (demo)" if RUN_DEMO else ""))


Frozen cohort: <REPOSITORY>/config/cohort_release_2026-11-13.csv
Selected samples: 43


## Prepare the preprocessing stage

In [4]:
stage_dir = paths.ensure_stage("01_Preprocessing")

if not RUN_DEMO:
    # Full rerun: replace this stage, but preserve 00_cohort and other stages.
    if stage_dir.exists():
        shutil.rmtree(stage_dir)
    stage_dir = paths.ensure_stage("01_Preprocessing")

executed_dir = stage_dir / "executed_notebooks"
log_dir = stage_dir / "logs"
tables_dir = stage_dir / "tables"
for directory in (executed_dir, log_dir, tables_dir):
    directory.mkdir(parents=True, exist_ok=True)

write_run_provenance(paths, stage_dir)

print(f"Preprocessing stage: {display_path(stage_dir)}")


Preprocessing stage: <CAP_DATA_ROOT>/runs/preprocessing_test_003/01_preprocessing


## Execute notebook 001 for each selected sample

In [5]:
completed_ids = []
failed_ids = []

for sample_id in sample_ids:
    executed_file = executed_dir / f"001_{sample_id}.ipynb"
    partial_file = executed_dir / f"001_{sample_id}.partial.ipynb"
    log_file = log_dir / f"{sample_id}.log"

    # Always rerun selected samples; do not skip because a previous output exists.
    executed_file.unlink(missing_ok=True)
    partial_file.unlink(missing_ok=True)

    print(f"Run {sample_id}")
    try:
        with log_file.open("w", encoding="utf-8") as log_handle:
            pm.execute_notebook(
                input_path=str(preprocessing_notebook),
                output_path=str(partial_file),
                parameters={"sample_id": sample_id},
                kernel_name=KERNEL_NAME,
                progress_bar=False,
                log_output=True,
                stdout_file=log_handle,
                stderr_file=log_handle,
                cwd=str(paths.root),
            )

        partial_file.replace(executed_file)
        completed_ids.append(sample_id)

    except Exception as exc:
        failed_ids.append(sample_id)
        print(
            f"FAILED {sample_id}: {type(exc).__name__}: {exc} "
            f"(log: {display_path(log_file)})"
        )

print(
    f"\nCompleted: {len(completed_ids)}  Failed: {len(failed_ids)}  "
    f"(of {len(sample_ids)} selected)"
)
if failed_ids:
    raise RuntimeError(
        f"{len(failed_ids)} sample(s) failed: {', '.join(failed_ids)}. "
        "See the per-sample logs and partial notebooks."
    )

Run GSM4909295_ER-AH0319


D:\Conda\envs\cap-rerun\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

R callback write-console: Creating ~2438 artificial doublets...
  

R callback write-console: Dimensional reduction
  

R callback write-console: Evaluating kNN...
  

R callback write-console: Training model...
  

R callback write-console: iter=0, 377 cells excluded from training.
  

R callback write-console: iter=1, 411 cells excluded from training.
  

R callback write-console: iter=2, 419 cells excluded from training.
  

R callback write-console: Threshold found:0.627
  

R callback write-console: 178 (5.8%) doublets called
  

D:\Conda\envs\cap-rerun\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)

D:\Conda\envs\cap-rerun\Lib\site-packages

Run GSM4909296_ER-MH0001


D:\Conda\envs\cap-rerun\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

R callback write-console: Creating ~5604 artificial doublets...
  

R callback write-console: Dimensional reduction
  

R callback write-console: Evaluating kNN...
  

R callback write-console: Training model...
  

R callback write-console: iter=0, 811 cells excluded from training.
  

R callback write-console: iter=1, 891 cells excluded from training.
  

R callback write-console: iter=2, 886 cells excluded from training.
  

R callback write-console: Threshold found:0.598
  

R callback write-console: 727 (10.4%) doublets called
  

D:\Conda\envs\cap-rerun\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)

D:\Conda\envs\cap-rerun\Lib\site-package

Run GSM4909302_ER-MH0025


D:\Conda\envs\cap-rerun\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

R callback write-console: Creating ~7420 artificial doublets...
  

R callback write-console: Dimensional reduction
  

R callback write-console: Evaluating kNN...
  

R callback write-console: Training model...
  

R callback write-console: iter=0, 1283 cells excluded from training.
  

R callback write-console: iter=1, 1366 cells excluded from training.
  

R callback write-console: iter=2, 1359 cells excluded from training.
  

R callback write-console: Threshold found:0.453
  

R callback write-console: 1073 (11.6%) doublets called
  

D:\Conda\envs\cap-rerun\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('scanpy')` instead
  mod_version = _find_version(mod.__version__)

D:\Conda\envs\cap-rerun\Lib\site-pac

... (77 outputs left out)

D:\Conda\envs\cap-rerun\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

R callback write-console: Creating ~1500 artificial doublets...
  

R callback write-console: Dimensional reduction
  

Autosave too slow: 9.34 sec, over 25% limit. Backing off to 60 sec
R callback write-console: Evaluating kNN...
  

R callback write-console: Training model...
  

R callback write-console: iter=0, 73 cells excluded from training.
  

R callback write-console: iter=1, 80 cells excluded from training.
  

R callback write-console: iter=2, 80 cells excluded from training.
  

R callback write-console: Threshold found:0.828
  

R callback write-console: 35 (3.7%) doublets called
  

D:\Conda\envs\cap-rerun\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_ve

Run GSM4909287_TN-B1-Tum0554


D:\Conda\envs\cap-rerun\Lib\site-packages\anndata\_core\anndata.py:1884: UserWarning: Variable names are not unique. To make them unique, call `.var_names_make_unique`.
  utils.warn_names_duplicates("var")

R callback write-console: Creating ~6680 artificial doublets...
  

R callback write-console: Dimensional reduction
  

R callback write-console: Evaluating kNN...
  

R callback write-console: Training model...
  

R callback write-console: iter=0, 1179 cells excluded from training.
  

R callback write-console: iter=1, 1397 cells excluded from training.
  

R callback write-console: iter=2, 1398 cells excluded from training.
  

R callback write-console: Threshold found:0.561
  

R callback write-console: 918 (11%) doublets called
  

D:\Conda\envs\cap-rerun\Lib\site-packages\session_info\main.py:213: FutureWarning: `__version__` is deprecated, use `importlib.metadata.version('anndata')` instead.
  mod_version = _find_version(mod.__version__)

D:\Conda\envs\cap-rerun\Lib\site-pack


Completed: 43  Failed: 0  (of 43 selected)


## Combine per-sample QC tables

In [6]:
qc_files = {
    sample_id: (
        tables_dir / "by_sample" / sample_id / "qc_summary.csv"
    )
    for sample_id in sample_ids
}
missing_qc = [sample_id for sample_id, path in qc_files.items() if not path.is_file()]
if missing_qc:
    missing_paths = [display_path(qc_files[sample_id]) for sample_id in missing_qc]
    raise FileNotFoundError(
        f"QC table(s) missing for {len(missing_qc)} sample(s): {missing_paths}"
    )

qc_summary = pd.concat(
    [pd.read_csv(qc_files[sample_id]) for sample_id in sample_ids],
    ignore_index=True,
)

if len(qc_summary) != len(sample_ids):
    raise ValueError(
        f"Expected one QC row per selected sample ({len(sample_ids)}), "
        f"found {len(qc_summary)} rows."
    )

sort_columns = [column for column in ("condition", "sample") if column in qc_summary]
if sort_columns:
    qc_summary = qc_summary.sort_values(sort_columns).reset_index(drop=True)

sample_column = next(
    (column for column in ("sample", "sample_id") if column in qc_summary),
    None,
)
if sample_column and qc_summary[sample_column].duplicated().any():
    raise ValueError(f"Duplicate sample rows in QC summary column {sample_column!r}.")

suffix = "_demo" if RUN_DEMO else ""
qc_summary_file = tables_dir / f"qc_summary{suffix}.csv"
qc_summary.to_csv(qc_summary_file, index=False, lineterminator="\n")

print(f"Written: {display_path(qc_summary_file)} ({len(qc_summary)} samples)")


Written: <CAP_DATA_ROOT>/runs/preprocessing_test_003/01_preprocessing/tables/qc_summary.csv (43 samples)
